In [5]:
import pandas as pd
 
# ─────────────────────────────────────────────
# STEP 0: LOAD THE DATASET
# ─────────────────────────────────────────────
df = pd.read_excel("Dataset for Data Analytics.xlsx")
 
print("=" * 55)
print("STEP 0: RAW DATASET OVERVIEW")
print("=" * 55)
print(f"Shape       : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns     : {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 5 rows:\n{df.head()}")

# ─────────────────────────────────────────────
# STEP 1: IDENTIFY MISSING VALUES
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 1: MISSING VALUES AUDIT")
print("=" * 55)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(missing_report[missing_report["Missing Count"] > 0])
 
 
# ─────────────────────────────────────────────
# PHASE 1: STRATEGIC IMPUTATION
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("PHASE 1: STRATEGIC IMPUTATION")
print("=" * 55)
 
# CouponCode has 309 nulls (25.8% missing)
# It is a categorical column — null simply means no coupon was used
# Strategy: Fill with "No Coupon" (domain logic, not statistical imputation)
before = df["CouponCode"].isnull().sum()
df["CouponCode"] = df["CouponCode"].fillna("No Coupon")
after = df["CouponCode"].isnull().sum()
 
print(f"CouponCode  : {before} nulls → filled with 'No Coupon' → {after} nulls remaining")
print(f"Unique CouponCode values: {df['CouponCode'].unique().tolist()}")
 
 
# ─────────────────────────────────────────────
# PHASE 2: THE INTEGRITY AUDIT (DUPLICATES)
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("PHASE 2: INTEGRITY AUDIT — DUPLICATES")
print("=" * 55)
 
dup_rows = df.duplicated().sum()
dup_ids  = df["OrderID"].duplicated().sum()
 
print(f"Duplicate rows     : {dup_rows}")
print(f"Duplicate OrderIDs : {dup_ids}")
 
if dup_rows > 0:
    print(f"→ Removing {dup_rows} duplicate rows...")
    df = df.drop_duplicates()
    print(f"  Done. New shape: {df.shape}")
else:
    print("→ No duplicates found. Dataset is clean on this check. ✓")
 
 
# ─────────────────────────────────────────────
# PHASE 3: SPEAK ONE LANGUAGE (STANDARDIZE FORMATS)
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("PHASE 3: FORMAT STANDARDIZATION")
print("=" * 55)
 
# --- 3a. Date → ISO 8601 (YYYY-MM-DD) ---
print("\n[3a] Date Format:")
print(f"  Before : {df['Date'].dtype} | sample: {df['Date'].head(2).tolist()}")
df["Date"] = pd.to_datetime(df["Date"]).dt.strftime("%Y-%m-%d")
print(f"  After  : {df['Date'].dtype} | sample: {df['Date'].head(2).tolist()} ✓")
 
# --- 3b. Text columns → Proper Case + Strip Whitespace ---
print("\n[3b] Text Columns — Proper Case & Trim Whitespace:")
text_cols = ["Product", "PaymentMethod", "OrderStatus",
             "ReferralSource", "ShippingAddress", "CouponCode",
             "OrderID", "CustomerID", "TrackingNumber"]
 
for col in text_cols:
    df[col] = df[col].str.strip().str.title()
    print(f"  {col:<20} → stripped & title-cased ✓")
 
# --- 3c. Numeric Precision → 2 decimal places ---
print("\n[3c] Numeric Precision (2 decimal places):")
df["UnitPrice"]  = df["UnitPrice"].round(2)
df["TotalPrice"] = df["TotalPrice"].round(2)
print(f"  UnitPrice  → rounded to 2dp ✓  sample: {df['UnitPrice'].head(3).tolist()}")
print(f"  TotalPrice → rounded to 2dp ✓  sample: {df['TotalPrice'].head(3).tolist()}")
 
 
# ─────────────────────────────────────────────
# VERIFICATION GATE (Project 2 Threshold)
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("VERIFICATION GATE")
print("=" * 55)
 
final_dup_ids = df["OrderID"].duplicated().sum()
bad_dates     = df["Date"].str.match(r"^\d{4}-\d{2}-\d{2}$") == False
bad_dates_count = bad_dates.sum()
 
print(f"Duplicate OrderIDs        : {final_dup_ids}  {'✓ PASS' if final_dup_ids == 0 else '✗ FAIL'}")
print(f"Incorrectly formatted dates: {bad_dates_count}  {'✓ PASS' if bad_dates_count == 0 else '✗ FAIL'}")
print(f"Remaining null values     : {df.isnull().sum().sum()}  {'✓ PASS' if df.isnull().sum().sum() == 0 else '✗ FAIL'}")
 
 
# ─────────────────────────────────────────────
# STEP 4: FINAL OVERVIEW
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 4: CLEANED DATASET OVERVIEW")
print("=" * 55)
print(f"Final Shape : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nSample of cleaned data:\n{df.head(5).to_string()}")
 
 
# ─────────────────────────────────────────────
# STEP 5: EXPORT CLEANED FILE
# ─────────────────────────────────────────────
output_file = "Cleaned_Dataset.xlsx"
df.to_excel(output_file, index=False)
print(f"\n✓ Cleaned dataset saved as: {output_file}")
 
 
# ─────────────────────────────────────────────
# CHANGE LOG (for your PDF report)
# ─────────────────────────────────────────────
print("\n" + "=" * 55)
print("CHANGE LOG SUMMARY")
print("=" * 55)
change_log = [
    ("CR001", "Phase 1", "CouponCode",   "309 null values filled with 'No Coupon'",                       "Preserved 309 records",        "Resolved"),
    ("CR002", "Phase 2", "All Columns",  "Checked for duplicate rows and duplicate OrderIDs",              "0 duplicates — verified clean", "Verified"),
    ("CR003", "Phase 3", "Date",         "Converted Date column to ISO 8601 format (YYYY-MM-DD)",          "1200 dates standardized",       "Resolved"),
    ("CR004", "Phase 3", "Text Columns", "Applied str.strip() and .str.title() to 9 text columns",        "Whitespace & casing unified",   "Resolved"),
    ("CR005", "Phase 3", "UnitPrice",    "Rounded UnitPrice to 2 decimal places",                         "Numeric precision enforced",    "Resolved"),
    ("CR006", "Phase 3", "TotalPrice",   "Rounded TotalPrice to 2 decimal places (float precision fix)",  "29 values corrected",           "Resolved"),
]
 
print(f"{'ID':<6} {'Phase':<9} {'Column':<16} {'Description':<52} {'Impact':<30} {'Status'}")
print("-" * 135)
for row in change_log:
    print(f"{row[0]:<6} {row[1]:<9} {row[2]:<16} {row[3]:<52} {row[4]:<30} {row[5]}")

STEP 0: RAW DATASET OVERVIEW
Shape       : 1200 rows × 14 columns
Columns     : ['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice', 'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber', 'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice']

Data Types:
OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object

First 5 rows:
     OrderID       Date CustomerID  Product  Quantity  UnitPrice  \
0  ORD200000 2023-01-04     C72649  Monitor         5     570.62   
1  ORD200001 2024-08-23     C75739    Phone         2     151.35  